<a id="analyse-descriptive"></a>

# Descriptive analysis: Verbal productions

<a id="longueur-productions"></a>

## SSD and Controls

Two length metrics are used in this notebook:

| Metric | Clear definition |
|---|---|
| **n_tokens** | **Total number of tokens** in the cleaned transcript (repetitions included, fillers normalized to `heu` and counted as tokens). This is the **gross production length**. |
| **n_mots** | **Number of distinct word types** in the cleaned transcript (unique vocabulary). This corresponds to the **number of speech-graph nodes**. |

In short: **n_tokens = how much is said** (quantity), while **n_mots = how varied the vocabulary is** (lexical diversity).

Fillers (e.g., euh, hein, bah) are normalized to `heu` during preprocessing. Hyphenated forms are split into separate words.

## Table of contents

1. [Production length (SSD vs CO)](#longueur-productions)
   - [Comparison by story (BD)](#comparaison-bd)
   - [Group effect: ANOVA at verbatim level](#anova)
   - [Linear mixed model (LMM)](#lmm)
2. [Backtracks in narration](#retours-en-arriere)
   - [Participant level](#niveau-participant)
   - [Generalized mixed model (GLMM)](#glmm)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, ks_2samp, levene, ttest_ind
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.stats.anova import anova_lm

mpl.rcParams['font.family'] = 'Times New Roman'
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

In [ ]:
DATA_FILE = '../SpeechGraph/data_SpeechGraph2.csv'

df_raw = pd.read_csv(DATA_FILE, sep=';')
df_raw.columns = df_raw.columns.str.strip()

df = df_raw[
    df_raw['Groupe'].isin([100, 300]) &
    (df_raw['n_tokens'] > 0)
].copy()
df = df.rename(columns={'No anonyme': 'No'})
df['Groupe'] = df['Groupe'].astype(int)
df['No'] = df['No'].astype(str).str.strip()

print(f'Verbatims group 100: {(df.Groupe==100).sum()}')
print(f'Verbatims group 300: {(df.Groupe==300).sum()}')

In [ ]:
# Check: 9 stories per participant + missing stories
check_df = df[['No', 'Groupe', 'BD']].copy()
check_df['story'] = check_df['BD'].astype(str).str.strip()
check_df = check_df[check_df['story'].notna() & (check_df['story'] != '')].copy()

# Expected reference: the 9 stories in the corpus
expected_stories = sorted(check_df['story'].unique())
print('Detected stories:', expected_stories)
print(f'Number of detected stories: {len(expected_stories)}')

rows = []
for (no, grp), sub in check_df.groupby(['No', 'Groupe']):
    observed = set(sub['story'].unique())
    missing = [s for s in expected_stories if s not in observed]
    rows.append({
        'No': no,
        'Groupe': grp,
        'n_histoires_uniques': len(observed),
        'missing_stories': ', '.join(missing) if missing else ''
    })

story_check = pd.DataFrame(rows).sort_values(['Groupe', 'No'])

print("\nParticipants with a number of stories different from 9:")
not_9 = story_check[story_check['n_histoires_uniques'] != 9]
if not_9.empty:
    print('None')
else:
    print(not_9[['No', 'Groupe', 'n_histoires_uniques', 'missing_stories']].to_string(index=False))

print(f"\nTotal participants checked: {len(story_check)}")
print(f"Participants with 9 stories: {(story_check['n_histoires_uniques'] == 9).sum()}")
print(f"Non-compliant participants: {(story_check['n_histoires_uniques'] != 9).sum()}")

In [ ]:
# Metrics loaded from precomputed CSV (output of visualization.ipynb)
# n_tokens : total number of tokens (gross speech length)
# n_nodes  → n_mots : number of unique words = graph nodes (lexical diversity)
df = df.rename(columns={'n_nodes': 'n_mots'})

ex = df.iloc[0]
print(f'Tokens (total length): {ex["n_tokens"]}  |  Distinct words (nodes): {ex["n_mots"]}')

In [ ]:
# Aggregation by participant (mean across all stories)
df_part = (
    df.groupby(['No', 'Groupe'])[['n_mots', 'n_tokens']]
    .mean()
    .reset_index()
)

print('Participants per group:')
print(df_part.groupby('Groupe')['No'].count())

print('\nDescriptive statistics:')
print(df_part.groupby('Groupe')[['n_mots', 'n_tokens']].describe().round(1))

In [ ]:
# Statistical tests
g100 = df_part[df_part['Groupe'] == 100]
g300 = df_part[df_part['Groupe'] == 300]

def cohens_d(a, b):
    s = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / s if s > 0 else np.nan

for col in ['n_mots', 'n_tokens']:
    a, b = g100[col].dropna(), g300[col].dropna()
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    d = cohens_d(a.values, b.values)
    print(f'{col}:')
    print(f'  SSD     (100): M={a.mean():.1f}  SD={a.std():.1f}  Md={a.median():.1f}')
    print(f'  Control (300): M={b.mean():.1f}  SD={b.std():.1f}  Md={b.median():.1f}')
    print(f"  Mann-Whitney U={U:.0f}  p={p:.4f}  Cohen's d={d:.3f}")
    print()

In [ ]:
# Visualization
labels = {100: 'SSD', 300: 'CO'}
colors = {100: '#E63946', 300: '#FF8C42'}
dot_colors = {100: '#E63946', 300: '#FF8C42'}

plot_specs = [
    ('n_mots', 'Number of words', 'Distinct words per transcript (participant mean)'),
    ('n_tokens', 'Number of tokens', 'Total number of words produced (participant mean)')
]

for col, title, ylab in plot_specs:
    fig, ax = plt.subplots(figsize=(6, 6))
    data = [g100[col].dropna().values, g300[col].dropna().values]
    bp = ax.boxplot(data, labels=[labels[100], labels[300]],
                    patch_artist=True, medianprops=dict(color='black', linewidth=2))
    for box, c in zip(bp['boxes'], [colors[100], colors[300]]):
        box.set_facecolor(c)
        box.set_alpha(0.75)
    for i, (d, c) in enumerate(zip(data, [dot_colors[100], dot_colors[100]]), 1):
        jitter = np.random.uniform(-0.08, 0.08, size=len(d))
        ax.scatter(i + jitter, d, alpha=0.55, s=24, color=c, zorder=3)

    U, p = stats.mannwhitneyu(data[0], data[1], alternative='two-sided')
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else f'p={p:.3f}'))
    ax.set_ylabel(ylab, fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.spines[['top']].set_visible(False)
    plt.tight_layout()
    plt.show()

**Conclusion – Overall production length (participant level)**

At the participant-aggregated level (n = 119), the two groups do not differ significantly in verbal production length:

Effect sizes are small (|d| < 0.3). Controls tend to produce slightly more words, but this trend does not reach significance with this test. A complementary analysis at the verbatim level (type II ANOVA) is presented below to assess whether this result holds when statistical power is increased.

In [ ]:
# See also by story
df_bd = df.groupby(['Groupe', 'BD'])[['n_mots', 'n_tokens']].mean().round(1)
print('Mean per group and story:')
print(df_bd)

<a id="comparaison-bd"></a>

## Comparison by story (BD)

Similar global means do not guarantee homogeneous distributions story by story. Three complementary tests:

| Test | What it checks |
|---|---|
| **t-test** | Difference in means between SSD and CO |
| **Levene** | Equality of variances (a difference in SD can be clinically important even without a mean difference) |
| **Kolmogorov-Smirnov** | Overall distribution shape (detects shape differences that the t-test misses) |

`*` = p < 0.05 &nbsp;|&nbsp; `(F)` = Fisher exact test used (expected count < 5)

In [ ]:
for col in ['n_mots', 'n_tokens']:
    print(f'=== {col} ===')
    print(f"{'BD':<10} {'M100':>6} {'SD100':>6} {'M300':>6} {'SD300':>6} | {'t(moy)':>8} {'p_t':>6} | {'Levene':>8} {'p_lev':>6} | {'KS':>6} {'p_ks':>6}")
    print('-' * 90)
    for bd in sorted(df['BD'].dropna().unique()):
        sub = df[df['BD'] == bd]
        a = sub[sub['Groupe'] == 100][col].dropna()
        b = sub[sub['Groupe'] == 300][col].dropna()
        if len(a) < 3 or len(b) < 3:
            continue
        t,   p_t   = ttest_ind(a, b)
        lev, p_lev = levene(a, b)
        ks,  p_ks  = ks_2samp(a, b)
        s_t   = '*' if p_t   < 0.05 else ' '
        s_lev = '*' if p_lev < 0.05 else ' '
        s_ks  = '*' if p_ks  < 0.05 else ' '
        print(f"{bd:<10} {a.mean():>6.1f} {a.std():>6.1f} {b.mean():>6.1f} {b.std():>6.1f} | {t:>8.2f} {p_t:>5.3f}{s_t} | {lev:>8.2f} {p_lev:>5.3f}{s_lev} | {ks:>6.3f} {p_ks:>5.3f}{s_ks}")
    print()

In [ ]:
# Figure: number of tokens by story with between-group significance
labels = {100: 'SSD', 300: 'CO'}
colors = {100: '#E63946', 300: '#FF8C42'}

# Order stories by level, then by story name
if 'Niveau' in df.columns and df['Niveau'].notna().any():
    story_levels = (
        df[['BD', 'Niveau']]
        .dropna()
        .copy()
    )
    story_levels['Niveau'] = pd.to_numeric(story_levels['Niveau'], errors='coerce')
    story_levels = story_levels.dropna(subset=['Niveau'])
    story_levels = (
        story_levels.groupby('BD', as_index=False)['Niveau']
        .median()
        .sort_values(['Niveau', 'BD'])
    )
    stories = story_levels['BD'].tolist()
    level_map = dict(zip(story_levels['BD'], story_levels['Niveau']))
else:
    stories = sorted(df['BD'].dropna().unique())
    level_map = {bd: np.nan for bd in stories}

# Mean and SEM per story/group
stats_tokens = (
    df.groupby(['BD', 'Groupe'])['n_tokens']
    .agg(['mean', 'sem'])
    .reset_index()
    .rename(columns={'mean': 'mean_tokens', 'sem': 'sem_tokens'})
)

fig, ax = plt.subplots(figsize=(9, 5))
fig.subplots_adjust(top=0.82)

x = np.arange(len(stories))

for grp, marker, lstyle in [(100, 'o', '-'), (300, 'D', '--')]:
    sub = stats_tokens[stats_tokens['Groupe'] == grp].set_index('BD').reindex(stories).reset_index()
    ax.errorbar(
        x,
        sub['mean_tokens'],
        yerr=sub['sem_tokens'],
        fmt=marker,
        linestyle='',
        color=colors[grp],
        capsize=4,
        linewidth=2,
        markersize=8,
        label=labels[grp],
        zorder=3
    )

# Add vertical separators between levels and level labels above the plot
level_sequence = [level_map.get(bd, np.nan) for bd in stories]
if not all(pd.isna(v) for v in level_sequence):
    separator_positions = []
    for i in range(1, len(stories)):
        prev_level = level_sequence[i - 1]
        curr_level = level_sequence[i]
        if pd.notna(prev_level) and pd.notna(curr_level) and curr_level != prev_level:
            ax.axvline(i - 0.5, color='black', linestyle='--', linewidth=2.5, zorder=4)
            separator_positions.append(i - 0.5)

    start = 0
    n_stories = len(stories)
    while start < n_stories:
        current_level = level_sequence[start]
        end = start
        while end + 1 < n_stories and level_sequence[end + 1] == current_level:
            end += 1
        if pd.notna(current_level):
            x_center_data = (start + end) / 2
            ax.annotate(
                f'Level {int(current_level)}',
                xy=(x_center_data, 1.0),
                xycoords=('data', 'axes fraction'),
                xytext=(0, 18),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=18,
                fontweight='bold',
                color='black',
                annotation_clip=False
            )
        start = end + 1

# Significance annotations per story
sig_count = 0
for i, bd in enumerate(stories):
    a = df[(df['BD'] == bd) & (df['Groupe'] == 100)]['n_tokens'].dropna()
    b = df[(df['BD'] == bd) & (df['Groupe'] == 300)]['n_tokens'].dropna()
    if len(a) < 3 or len(b) < 3:
        continue
    _, p_ks = ks_2samp(a, b)
    if p_ks < 0.05:
        sig = '***' if p_ks < 0.001 else ('**' if p_ks < 0.01 else '*')
        y = max(a.mean(), b.mean()) + 8
        ax.text(i, y, f'{sig}', ha='center', va='bottom', fontsize=12, fontweight='bold')
        sig_count += 1

ax.set_xticks(x)
ax.set_xticklabels(stories, rotation=25, ha='right', fontsize=18)
ax.set_ylabel('Number of tokens (mean ± SEM)', fontsize=18)
ax.set_xlabel('Story (ordered by referential complexity)', fontsize=18)
ax.legend(title='Group', fontsize=18, title_fontsize=18, frameon=True, edgecolor='black', fancybox=False)
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

if sig_count == 0:
    ax.text(
        0.01, 0.98,
        'No story shows a significant between-group KS difference in n_tokens (p < 0.05).',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=15,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

plt.tight_layout()
plt.show()

**Conclusion – Comparison by story**

The story-by-story breakdown confirms the absence of a systematic difference between SSD and Controls in production length:

- **n_mots**: only the story *MiseB* shows a significant mean difference (t-test, p = 0.035), with controls producing more distinct words. Stories *Bottes* and *Jongle* show distribution shape differences (KS, p < 0.05) without mean differences.
- **n_tokens**: no story shows a significant mean difference. Only *Jongle* shows a distribution difference (KS, p = 0.021).
- Levene tests reveal no significant variance heterogeneity between the two groups for any story.

Overall, both groups produce narratives of comparable length, with isolated effects on certain stories that do not survive correction for multiple comparisons.

<a id="anova"></a>

## Group effect on production length: verbatim-level analysis (ANOVA)

The previous Mann-Whitney test aggregates each participant into **a single mean** (across their 9 stories), reducing the sample to n = 119. To gain statistical power, the data can be analyzed at the **verbatim level** (n = 1,071) with a two-way ANOVA:

- **C(Groupe)**: SSD vs Controls
- **C(BD)**: the 9 stories
- **C(Groupe) × C(BD)**: the interaction — tests whether the group difference depends on the story

Type II ANOVA absorbs the between-story variance into the `C(BD)` term, which **reduces residual error** and increases sensitivity to detect the group effect.

In [ ]:
for col in ['n_mots', 'n_tokens']:
    lm = smf.ols(f'{col} ~ C(Groupe) * C(BD)', data=df).fit()
    aov = anova_lm(lm, typ=2)

    print('=' * 70)
    print(f'TYPE II ANOVA: {col} ~ C(Groupe) * C(BD)')
    print('=' * 70)
    print(f'\n{"Source":<25} {"SS":>12} {"df":>5} {"F":>8} {"p":>8}')
    print('-' * 62)
    for src in aov.index:
        row = aov.loc[src]
        F_val = row.get('F', float('nan'))
        p_val = row.get('PR(>F)', float('nan'))
        sig = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
        print(f'{src:<25} {row["sum_sq"]:>12.1f} {row["df"]:>5.0f} '
              f'{F_val:>8.2f} {p_val:>7.4f}{sig}')

    p_inter = aov.loc['C(Groupe):C(BD)', 'PR(>F)']
    p_groupe = aov.loc['C(Groupe)', 'PR(>F)']
    p_bd = aov.loc['C(BD)', 'PR(>F)']
    print(f'\n→ Group effect: F = {aov.loc["C(Groupe)", "F"]:.2f}, p = {p_groupe:.4f}')
    print(f'→ Story effect: F = {aov.loc["C(BD)", "F"]:.2f}, p = {p_bd:.4f}')
    print(f'→ Interaction:  F = {aov.loc["C(Groupe):C(BD)", "F"]:.2f}, p = {p_inter:.4f}')
    print()

**Conclusion – Verbatim-level ANOVA vs. participant-level Mann-Whitney**

Both analyses detect the same trend (controls produce slightly more words than SSD), but reach different statistical conclusions:

| Approach | Unit of analysis | n | Result (n_mots) |
|---|---|---|---|
| Mann-Whitney (previous section) | Mean per **participant** | 119 | p = 0.170 (NS) |
| Type II ANOVA (above) | Each **verbatim** | 1,071 | p = 0.0001 (**significant**) |

**Why this divergence?**

1. **Power loss by aggregation**: summarizing each participant by their mean across 9 stories reduces from 1,071 to 119 observations. The Mann-Whitney test therefore has far less data to detect a moderate effect.
2. **Absorption of between-story variance**: the ANOVA isolates the story effect (`C(BD)`, F = 19.70) in a dedicated term. Variance due to difficulty differences between stories is removed from residual error, increasing sensitivity for the Group factor.
3. **No Group × Story interaction**: the interaction terms are far from significance (p ≈ 0.99), confirming that the SSD vs CO gap is **homogeneous across the 9 stories**.

**Interpretation**: the participant-level aggregation (Mann-Whitney) is more conservative and traditionally preferred for between-group comparisons, as it respects observation independence. The verbatim-level ANOVA is more sensitive but treats observations from the same participant as independent, which can inflate false positives. A **linear mixed model** (with participant as random effect) would be the ideal compromise between power and rigor.

<a id="lmm"></a>

## Linear mixed model (LMM)

The verbatim-level ANOVA artificially inflates *n* by treating observations from the same participant as independent. The **linear mixed model** solves this by adding participant as a **random effect** (random intercept per subject). It thus combines:

- the power of the verbatim level (n = 1,071),
- control for within-participant non-independence,
- control for the story effect.

Model: `metric ~ C(Groupe) + C(BD)`, with random intercept per participant (`groups = No`).

In [ ]:
for col in ['n_mots', 'n_tokens']:
    lmm = smf.mixedlm(f'{col} ~ C(Groupe) + C(BD)', data=df, groups=df['No']).fit()

    print('=' * 70)
    print(f'LMM: {col} ~ C(Groupe) + C(BD), random intercept per participant')
    print('=' * 70)

    coef_grp = lmm.params['C(Groupe)[T.300]']
    p_grp = lmm.pvalues['C(Groupe)[T.300]']
    ci_lo, ci_hi = lmm.conf_int().loc['C(Groupe)[T.300]']
    sig = '*' if p_grp < 0.05 else ''

    print(f'\n  Group effect (300 vs 100):')
    print(f'    β = {coef_grp:.2f}  [95% CI: {ci_lo:.2f} ; {ci_hi:.2f}]')
    print(f'    z = {lmm.tvalues["C(Groupe)[T.300]"]:.2f},  p = {p_grp:.4f}{sig}')
    print(f'\n  Random intercept variance (participant): '
          f'{lmm.cov_re.iloc[0, 0]:.1f}')
    print(f'  Residual variance: {lmm.scale:.1f}')
    print()

**Conclusion – Linear mixed model**

The LMM confirms the absence of a significant difference between groups once within-participant non-independence is properly modeled:

| Metric | β (CO vs SSD) | 95% CI | z | p |
|---|---|---|---|---|
| **n_mots** | +4.47 | [−1.24; 10.17] | 1.53 | 0.125 |
| **n_tokens** | +7.47 | [−5.46; 20.39] | 1.13 | 0.258 |

The random intercept variance (participant) is high relative to residual variance (n_mots: 233.9 vs 117.4; n_tokens: 1,193.7 vs 663.7), confirming that productions vary more **between participants** than between verbatims from the same participant.

**Summary of the three approaches:**

| Approach | Level | n | p (n_mots) | p (n_tokens) |
|---|---|---|---|---|
| Mann-Whitney | Participant | 119 | 0.170 | 0.414 |
| Type II ANOVA | Verbatim | 1,071 | 0.0001 | 0.013 |
| **LMM** | **Verbatim + random effect** | **1,071** | **0.125** | **0.258** |

The verbatim-level ANOVA overestimated significance by ignoring within-participant correlation. The LMM, which is the most appropriate model for these data (repeated measures, hierarchical structure), confirms the Mann-Whitney result: **the two groups do not differ significantly in verbal production length**.